<a href="https://colab.research.google.com/github/nexageapps/AI/blob/main/Basic/B11%20-%20Attention%20and%20Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 11: Attention Mechanisms & Transformers

**Author:** Karthik Arjun  
**LinkedIn:** [karthik-arjun-a5b4a258](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Created:** 2026  
**Updated:** 2026

## Learning Objectives
- Understand the attention mechanism
- Learn self-attention and its importance
- Master multi-head attention
- Understand Transformer architecture
- Implement positional encoding

## Prerequisites
- Completed L1-L10
- Understanding of RNNs and sequence modeling

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
import seaborn as sns

print(f"TensorFlow version: {tf.__version__}")

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

## 1. The Problem with RNNs

### RNN Limitations:
1. **Sequential processing:** Can't parallelize
2. **Long-range dependencies:** Still struggle despite LSTM/GRU
3. **Fixed context:** Hidden state is a bottleneck

### Example:
"The **animal** didn't cross the street because **it** was too tired."

What does "it" refer to? The animal or the street?

**Attention** helps the model focus on relevant parts of the input!

## 2. Understanding Attention

### Core Idea:
Instead of compressing entire sequence into fixed vector,
**attend** to different parts based on what's relevant.

### Attention Mechanism:
```
1. Query (Q): What am I looking for?
2. Key (K): What do I have?
3. Value (V): What information do I provide?

Attention(Q, K, V) = softmax(QK^T / √d_k) V
```

### Intuition:
- **Similarity:** Compare query with all keys
- **Weights:** Softmax to get attention weights
- **Weighted sum:** Combine values based on weights

In [ ]:
# Implement simple attention mechanism
def scaled_dot_product_attention(query, key, value, mask=None):
    """
    Calculate attention weights and apply to values.
    
    Args:
        query: Query tensor (batch_size, seq_len, d_k)
        key: Key tensor (batch_size, seq_len, d_k)
        value: Value tensor (batch_size, seq_len, d_v)
        mask: Optional mask tensor
    
    Returns:
        output: Attention output
        attention_weights: Attention weights
    """
    # Calculate attention scores
    d_k = tf.cast(tf.shape(key)[-1], tf.float32)
    scores = tf.matmul(query, key, transpose_b=True) / tf.math.sqrt(d_k)
    
    # Apply mask if provided
    if mask is not None:
        scores += (mask * -1e9)
    
    # Calculate attention weights
    attention_weights = tf.nn.softmax(scores, axis=-1)
    
    # Apply attention to values
    output = tf.matmul(attention_weights, value)
    
    return output, attention_weights

# Demonstrate with example
batch_size = 1
seq_len = 5
d_model = 8

# Create random Q, K, V
query = tf.random.normal((batch_size, seq_len, d_model))
key = tf.random.normal((batch_size, seq_len, d_model))
value = tf.random.normal((batch_size, seq_len, d_model))

# Calculate attention
output, attention_weights = scaled_dot_product_attention(query, key, value)

print(f"Query shape: {query.shape}")
print(f"Key shape: {key.shape}")
print(f"Value shape: {value.shape}")
print(f"\nOutput shape: {output.shape}")
print(f"Attention weights shape: {attention_weights.shape}")

# Visualize attention weights
plt.figure(figsize=(8, 6))
sns.heatmap(attention_weights[0].numpy(), annot=True, fmt='.2f', cmap='YlOrRd')
plt.title('Attention Weights', fontsize=14)
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.show()

print("\nEach row shows how much attention each query pays to each key.")
print("Rows sum to 1.0 (softmax property).")

## 3. Self-Attention

**Self-attention:** Query, Key, and Value all come from the same sequence.

### Why Self-Attention?
- Each word attends to all other words
- Captures relationships within the sequence
- Parallel processing (no sequential dependency)

### Example:
"The animal didn't cross the street because it was too tired."

Self-attention helps "it" attend strongly to "animal".

In [ ]:
# Demonstrate self-attention on a sentence
sentence = "The cat sat on the mat"
words = sentence.split()

# Create simple word embeddings (random for demo)
vocab_size = len(words)
d_model = 16
embeddings = tf.random.normal((1, vocab_size, d_model))

# Self-attention: Q, K, V all from same embeddings
output, attention_weights = scaled_dot_product_attention(
    embeddings, embeddings, embeddings
)

# Visualize
plt.figure(figsize=(8, 6))
sns.heatmap(attention_weights[0].numpy(), 
            annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=words, yticklabels=words)
plt.title('Self-Attention Weights', fontsize=14)
plt.xlabel('Attending to')
plt.ylabel('Word')
plt.tight_layout()
plt.show()

print("Each word attends to all words in the sentence.")
print("Diagonal shows self-attention (word attending to itself).")

## 4. Multi-Head Attention

Instead of one attention mechanism, use **multiple heads**:
- Each head learns different relationships
- Head 1: Syntactic relationships
- Head 2: Semantic relationships
- Head 3: Long-range dependencies

### Process:
1. Project Q, K, V into multiple subspaces
2. Apply attention in each subspace
3. Concatenate results
4. Final linear projection

In [ ]:
# Implement Multi-Head Attention
class MultiHeadAttention(layers.Layer):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        
        assert d_model % num_heads == 0
        
        self.depth = d_model // num_heads
        
        self.wq = layers.Dense(d_model)
        self.wk = layers.Dense(d_model)
        self.wv = layers.Dense(d_model)
        
        self.dense = layers.Dense(d_model)
    
    def split_heads(self, x, batch_size):
        """Split the last dimension into (num_heads, depth)."""
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])
    
    def call(self, v, k, q, mask=None):
        batch_size = tf.shape(q)[0]
        
        # Linear projections
        q = self.wq(q)
        k = self.wk(k)
        v = self.wv(v)
        
        # Split into multiple heads
        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)
        
        # Scaled dot-product attention
        scaled_attention, attention_weights = scaled_dot_product_attention(
            q, k, v, mask
        )
        
        # Concatenate heads
        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(scaled_attention, 
                                     (batch_size, -1, self.d_model))
        
        # Final linear projection
        output = self.dense(concat_attention)
        
        return output, attention_weights

# Test multi-head attention
d_model = 64
num_heads = 8
seq_len = 10
batch_size = 2

mha = MultiHeadAttention(d_model, num_heads)

# Create sample input
x = tf.random.normal((batch_size, seq_len, d_model))

# Apply multi-head attention
output, attention_weights = mha(x, x, x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attention_weights.shape}")
print(f"\nNumber of heads: {num_heads}")
print(f"Depth per head: {d_model // num_heads}")

## 5. Positional Encoding

**Problem:** Attention has no notion of position/order.

**Solution:** Add positional information to embeddings.

### Sinusoidal Positional Encoding:
```
PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

Where:
- pos: Position in sequence
- i: Dimension index

In [ ]:
# Implement positional encoding
def get_positional_encoding(seq_len, d_model):
    """
    Generate positional encoding.
    
    Args:
        seq_len: Sequence length
        d_model: Model dimension
    
    Returns:
        Positional encoding tensor
    """
    position = np.arange(seq_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    
    pos_encoding = np.zeros((seq_len, d_model))
    pos_encoding[:, 0::2] = np.sin(position * div_term)
    pos_encoding[:, 1::2] = np.cos(position * div_term)
    
    return tf.cast(pos_encoding[np.newaxis, ...], dtype=tf.float32)

# Generate and visualize positional encoding
seq_len = 50
d_model = 128

pos_encoding = get_positional_encoding(seq_len, d_model)

plt.figure(figsize=(12, 6))
plt.pcolormesh(pos_encoding[0].numpy(), cmap='RdBu')
plt.xlabel('Dimension', fontsize=12)
plt.ylabel('Position', fontsize=12)
plt.colorbar(label='Value')
plt.title('Positional Encoding', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Positional encoding shape: {pos_encoding.shape}")
print("\nProperties:")
print("- Each position has unique encoding")
print("- Sinusoidal pattern allows model to learn relative positions")
print("- Can extrapolate to longer sequences than seen during training")

## 6. Transformer Encoder Block

A Transformer encoder block consists of:
1. **Multi-Head Self-Attention**
2. **Add & Norm** (Residual connection + Layer Normalization)
3. **Feed-Forward Network**
4. **Add & Norm**

In [ ]:
# Implement Transformer Encoder Block
class TransformerEncoderBlock(layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super(TransformerEncoderBlock, self).__init__()
        
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = keras.Sequential([
            layers.Dense(dff, activation='relu'),
            layers.Dense(d_model)
        ])
        
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)
    
    def call(self, x, training, mask=None):
        # Multi-head attention
        attn_output, _ = self.mha(x, x, x, mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)  # Residual connection
        
        # Feed-forward network
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)  # Residual connection
        
        return out2

# Test encoder block
d_model = 64
num_heads = 8
dff = 256
seq_len = 10
batch_size = 2

encoder_block = TransformerEncoderBlock(d_model, num_heads, dff)

# Create sample input
x = tf.random.normal((batch_size, seq_len, d_model))

# Apply encoder block
output = encoder_block(x, training=False)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print("\nEncoder block maintains input shape!")

## 7. Complete Transformer Encoder

Stack multiple encoder blocks to create a complete Transformer encoder.

In [ ]:
# Build complete Transformer Encoder
class TransformerEncoder(layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, dff, 
                 vocab_size, max_seq_len, dropout_rate=0.1):
        super(TransformerEncoder, self).__init__()
        
        self.d_model = d_model
        self.num_layers = num_layers
        
        self.embedding = layers.Embedding(vocab_size, d_model)
        self.pos_encoding = get_positional_encoding(max_seq_len, d_model)
        
        self.enc_layers = [
            TransformerEncoderBlock(d_model, num_heads, dff, dropout_rate)
            for _ in range(num_layers)
        ]
        
        self.dropout = layers.Dropout(dropout_rate)
    
    def call(self, x, training, mask=None):
        seq_len = tf.shape(x)[1]
        
        # Embedding + positional encoding
        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x += self.pos_encoding[:, :seq_len, :]
        
        x = self.dropout(x, training=training)
        
        # Pass through encoder layers
        for enc_layer in self.enc_layers:
            x = enc_layer(x, training, mask)
        
        return x

# Build Transformer for text classification
num_layers = 4
d_model = 128
num_heads = 8
dff = 512
vocab_size = 10000
max_seq_len = 200
num_classes = 2

# Create model
inputs = layers.Input(shape=(None,), dtype=tf.int32)
encoder = TransformerEncoder(num_layers, d_model, num_heads, dff, 
                            vocab_size, max_seq_len)
x = encoder(inputs, training=False)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model_transformer = keras.Model(inputs=inputs, outputs=outputs, 
                               name='Transformer_Classifier')

model_transformer.summary()

## 8. Key Takeaways

### Attention Mechanism:
1. **Attention** allows models to focus on relevant parts
2. **Self-attention** captures relationships within sequence
3. **Multi-head attention** learns different types of relationships

### Transformers:
4. **No recurrence:** Fully parallel processing
5. **Positional encoding:** Adds position information
6. **Residual connections:** Help with gradient flow
7. **Layer normalization:** Stabilizes training

### Advantages:
8. **Parallelizable:** Much faster training than RNNs
9. **Long-range dependencies:** Better than RNNs
10. **State-of-the-art:** Foundation for GPT, BERT, etc.

## Next Steps (L12)
- Learn Byte Pair Encoding (BPE) tokenization
- Understand subword tokenization
- Prepare for building language models

## References
- Attention Is All You Need: https://arxiv.org/abs/1706.03762
- The Illustrated Transformer: http://jalammar.github.io/illustrated-transformer/
- TensorFlow Transformer Tutorial: https://www.tensorflow.org/text/tutorials/transformer

---

## Key Takeaways

**Relevant UoA Courses:** COMPSCI 703, COMPSYS 721, COMPSCI 714

1. Attention: Attention(Q,K,V) = softmax(QK^T/√d_k)V
2. Self-attention: each token attends to all other tokens
3. Multi-head attention: multiple attention mechanisms in parallel
4. Transformers: encoder-decoder with self-attention (no recurrence)
5. Positional encoding adds position information (since no recurrence)

---

## Exam Preparation Guide

### Essential Concepts for Exams

- Calculate attention scores: given Q, K, compute QK^T/√d_k then softmax
- Understand why scale by √d_k: prevents softmax saturation
- Know transformer architecture: Multi-head attention → Add&Norm → FFN → Add&Norm
- Explain why transformers are better than RNNs: parallelization, long-range dependencies
- Common question: Given Q(2,3), K(4,3), V(4,5), what's attention output shape?

### Common Mistakes to Avoid

- ❌ Forgetting to scale by √d_k in attention
- ❌ Confusing encoder and decoder attention mechanisms
- ❌ Not understanding that transformers have no inherent position information

### Practice Problems

1. Q=(2,64), K=(10,64), V=(10,128). Calculate attention output shape
2. Why divide by √d_k in attention formula?
3. Compare transformers vs RNNs: parallelization, memory, long-range dependencies

### How This Helps Your UoA Courses

**COMPSCI 703, COMPSYS 721, COMPSCI 714:**
- Provides hands-on implementation of theoretical concepts
- Practice problems similar to exam questions
- Reinforces lecture material with code examples
- Helps build intuition for complex topics

### Study Tips

1. **Understand, Don't Memorize**: Focus on why, not just what
2. **Practice Calculations**: Work through problems by hand
3. **Connect to Theory**: Link code to mathematical formulations
4. **Teach Others**: Explaining concepts solidifies understanding
5. **Review Regularly**: Spaced repetition improves retention

### Exam Question Types

- **Conceptual**: Explain why/how something works
- **Calculation**: Compute outputs, gradients, shapes
- **Comparison**: Compare approaches, trade-offs
- **Application**: Design solution for given problem
- **Debugging**: Identify and fix issues

---


---

## Learning Progress Tracker

Use this section to track your learning progress for this lesson.

### Completion Status
- [ ] Lesson completed
- [ ] All code cells executed successfully
- [ ] Understood all key concepts
- [ ] Completed practice exercises (if any)

### Dates
- **First Completed:** ____/____/____
- **Last Reviewed:** ____/____/____
- **Next Review:** ____/____/____ (Recommended: 1 week, 1 month, 3 months)

### Understanding Level
Rate your understanding (1-5): _____ / 5

- 1 = Need to review completely
- 2 = Understood basics, need more practice
- 3 = Good understanding, minor gaps
- 4 = Strong understanding, can explain to others
- 5 = Mastered, can apply in projects

### Notes & Reflections
```
Write your notes here:
- What concepts were challenging?
- What was interesting or surprising?
- How can you apply this in projects?
- Questions to explore further?




```

### Key Concepts to Remember (B11)
- [ ] Self-attention mechanism
- [ ] Multi-head attention
- [ ] Transformer architecture
- [ ] Positional encoding

---